# 02 - Data Cleaning Pipeline
## ShopEase Europe | Sentiment Analysis Project 
**Objective:** Clean and standardise the raw customer reviews dataset 
based on findings from the Data Quality Assessment. Produce a clean, 
analysis-ready dataset saved to data/processed/.

### Cleaning Actions Based On Audit Findings
- Convert timestamp to datetime format
- Standardise text casing and whitespace
- Remove noise from review text - special characters, excessive punctuation
- Handle emojis - retain sentiment-bearing ones, remove decorative noise
- Flag and document duplicate review texts
- Save cleaned dataset to data/processed/clean_reviews.csv

In [1]:
import os
import re
import warnings
import pandas as pd
import numpy as np
warnings.filterwarnings('ignore')

# Project paths
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
RAW_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'raw', 'raw_reviews.csv')
PROCESSED_DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'clean_reviews.csv')

print("Libraries loaded successfully")
print(f"Raw data: {RAW_DATA_PATH}")
print(f"Output path: {PROCESSED_DATA_PATH}")
print(f"Raw data exists: {os.path.exists(RAW_DATA_PATH)}")

Libraries loaded successfully
Raw data: c:\Users\User\OneDrive\Desktop\shopease-sentiment-analysis\data\raw\raw_reviews.csv
Output path: c:\Users\User\OneDrive\Desktop\shopease-sentiment-analysis\data\processed\clean_reviews.csv
Raw data exists: True


## 2.0 Load Raw Dataset

In [2]:
# Load raw dataset
df = pd.read_csv(RAW_DATA_PATH)
print(f"Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
display(df.head())

Dataset loaded: 120,000 rows x 7 columns


,review_id,product_category,timestamp,country,rating,review,sentiment
0,1,Books,2023-01-01,GB,3,"Solid build, attractive design, works as adver...",positive
1,2,Toys,2023-01-01,DE,5,Ich liebe dieses Produkt! ⭐⭐⭐,positive
2,3,Beauty,2023-01-01,AU,3,Three stars — meets THE minimum expectations.,neutral
3,4,Toys,2023-01-01,US,5,"Solid build, attractive design, works as adver...",positive
4,5,Electronics,2023-01-01,CA,2,Broken on arrival. Return process was a NIGHTM...,negative


## 3.0 Structural Cleaning
Fix data types and standardise structural columns before touching the review text.

In [3]:
# Convert timestamp to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Extract useful time features for later analysis
df['year'] = df['timestamp'].dt.year
df['month'] = df['timestamp'].dt.month
df['month_name'] = df['timestamp'].dt.strftime('%B')

# Standardise country and category to uppercase and stripped
df['country'] = df['country'].str.strip().str.upper()
df['product_category'] = df['product_category'].str.strip().str.title()

# Confirm changes
print("STRUCTURAL CLEANING COMPLETE")
print(f"Timestamp dtype:     {df['timestamp'].dtype}")
print(f"Unique countries:    {df['country'].nunique()}")
print(f"Unique categories:   {df['product_category'].nunique()}")
print(f"Year range:          {df['year'].min()} - {df['year'].max()}")
display(df[['timestamp', 'year', 'month', 'country', 'product_category']].head())

STRUCTURAL CLEANING COMPLETE
Timestamp dtype:     datetime64[ns]
Unique countries:    14
Unique categories:   8
Year range:          2023 - 2025


,timestamp,year,month,country,product_category
0,2023-01-01,2023,1,GB,Books
1,2023-01-01,2023,1,DE,Toys
2,2023-01-01,2023,1,AU,Beauty
3,2023-01-01,2023,1,US,Toys
4,2023-01-01,2023,1,CA,Electronics


## 4.0 Text Cleaning Pipeline
Clean the raw review text by removing noise while preserving 
sentiment-bearing content. Cleaning is applied per language 
to avoid incorrectly stripping valid characters.

In [4]:
def clean_text(text):
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove excessive punctuation — keep single instances
    text = re.sub(r'[!]{2,}', '!', text)
    text = re.sub(r'[?]{2,}', '?', text)
    text = re.sub(r'[.]{2,}', '.', text)
    
    # Remove special characters but keep core punctuation
    text = re.sub(r'[^a-zA-ZÀ-ÿ0-9\s\.,!?\'-]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning to review column
df['cleaned_review'] = df['review'].apply(clean_text)

# Verify cleaning worked
print("TEXT CLEANING COMPLETE")
print(f"Sample original:  {df['review'].iloc[4]}")
print(f"Sample cleaned:   {df['cleaned_review'].iloc[4]}")
print(f"\nOriginal length avg:  {df['review'].str.len().mean():.1f} characters")
print(f"Cleaned length avg:   {df['cleaned_review'].str.len().mean():.1f} characters")

TEXT CLEANING COMPLETE
Sample original:  Broken on arrival. Return process was a NIGHTMARE.
Sample cleaned:   broken on arrival. return process was a nightmare.

Original length avg:  53.6 characters
Cleaned length avg:   53.3 characters
